# Phase 4 — VideoMAE Fight Detection Fine-tuning
- Model: MCG-NJU/videomae-base
- Dataset: RWF-2000 (1600 train, 400 val)
- Device: T4 GPU (Colab)


In [ ]:
# Setup
!pip install torch torchvision transformers datasets scikit-learn supervision ultralytics


In [ ]:
# Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

# The project pins requires-python >=3.12,<3.13. If `pip install -e .` below complains about the
# Python version, skip it: the sys.path insert in the Train cell is enough to import drishti.
!test -d drishti || git clone https://github.com/ParrvLuthra22/drishti.git
%cd drishti
!pip install -e .


In [ ]:
# Upload dataset
# Option A: if RWF-2000 is in Google Drive
import shutil, os
src = '/content/drive/MyDrive/RWF-2000'
dst = 'data/rwf2000'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied from Drive")
else:
    print("Upload RWF-2000 manually or mount Drive")

# Expect data/rwf2000/{train,val}/{Fight,NonFight}/*.avi
for split in ('train', 'val'):
    for cls in ('Fight', 'NonFight'):
        folder = f'{dst}/{split}/{cls}'
        print(folder, len(os.listdir(folder)) if os.path.isdir(folder) else 'MISSING')


In [ ]:
# Verify GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
!nvidia-smi


In [ ]:
# Train
import sys
sys.path.insert(0, '/content/drishti/src')
from drishti.actions.trainer import VideoMAETrainer

trainer = VideoMAETrainer(
    data_root='data/rwf2000',
    output_dir='models/videomae_fight',
    device='cuda',
    batch_size=4,
    num_epochs=5
)
trainer.train()


In [ ]:
# Push to HuggingFace Hub
# Store a write token in Colab Secrets (key icon in the left sidebar) as HF_TOKEN, and enable
# notebook access. Do not paste the token into this notebook: it gets committed to GitHub.
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path='models/videomae_fight',
    repo_id='ParrvLuthra22/drishti-fight-detector',
    repo_type='model',
    token=userdata.get('HF_TOKEN')
)
print("Pushed to HuggingFace Hub")


In [ ]:
# Quick eval
trainer.evaluate('models/videomae_fight/best_model.pt')
